# IF AF3 Benchmark Analysis

This notebook analyzes `/home/sujin/projects/cdr-scoring/cdr-data/inference/IF-af3-benchmark.csv`, computes missing structural metrics on demand, and compares SJscore against the raw AF3 `af3_ranking_score`. It avoids target-specific filtering and uses a cache so expensive metric calculation is only done once per model.

In [ ]:
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
import csv
import math
import os
import sys
import time
import warnings

REPO_ROOT = Path('/home/sujin/projects/cdr-scoring/cdr-code')
OUT_DIR = REPO_ROOT / 'notebooks' / 'tmp'
PLOT_DIR = OUT_DIR / 'plots_if_af3_benchmark'
CACHE_PATH = OUT_DIR / 'IF-af3-benchmark.metrics_cache.csv'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / '.cache').mkdir(parents=True, exist_ok=True)
(OUT_DIR / '.matplotlib').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('XDG_CACHE_HOME', str(OUT_DIR / '.cache'))
os.environ.setdefault('MPLCONFIGDIR', str(OUT_DIR / '.matplotlib'))

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

if str(REPO_ROOT / 'libs') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'libs'))

DATA_PATH = Path('/home/sujin/projects/cdr-scoring/cdr-data/inference/IF-af3-benchmark.csv')

# Full default: compute metrics for all decoys so all downstream plots use all 18,500 rows.
# Set COMPUTE_ALL_METRICS=False for a quick exploratory run on top-N SJscore/AF3-score decoys only.
COMPUTE_ALL_METRICS = True
TOP_N_FOR_METRICS = 10
SUCCESS_RMSD_CUTOFF = 2.0
SUCCESS_LDDT_CUTOFF = 0.8
MAX_WORKERS = min(12, os.cpu_count() or 1)

METRIC_COLS = ['loop_rmsd', 'loop_lddt', 'irmsd', 'lrmsd']
LOWER_IS_BETTER = {'loop_rmsd': True, 'loop_lddt': False, 'irmsd': True, 'lrmsd': True}

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})
warnings.filterwarnings('ignore', category=RuntimeWarning)


def as_float(value):
    try:
        x = float(value)
    except (TypeError, ValueError):
        return np.nan
    return x


def finite_array(rows, col):
    arr = np.array([as_float(r.get(col)) for r in rows], dtype=float)
    return arr[np.isfinite(arr)]


def group_rows(rows, key='target_id'):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row[key]].append(row)
    return grouped


def metric_key(row):
    return '||'.join(str(row.get(k, '')) for k in ['target_id', 'sample_id', 'native_path', 'path'])

In [ ]:
with DATA_PATH.open(newline='') as f:
    rows = list(csv.DictReader(f))

numeric_cols = [
    'rank_by_model', 'pred_score', 'seed', 'sample', 'af3_rank', 'af3_ranking_score',
    'loop_rmsd', 'loop_lddt', 'irmsd', 'lrmsd', 'n_nodes', 'n_edges',
]
for row in rows:
    for col in numeric_cols:
        if col in row:
            row[col] = as_float(row[col])

targets = group_rows(rows)
for target_id, target_rows in targets.items():
    for rank, row in enumerate(sorted(target_rows, key=lambda r: as_float(r['pred_score'])), start=1):
        row['sj_rank'] = rank
    for rank, row in enumerate(sorted(target_rows, key=lambda r: as_float(r['af3_ranking_score']), reverse=True), start=1):
        row['af3_rank_from_score'] = rank
    for row in target_rows:
        row['rank_delta_sj_minus_af3_score'] = as_float(row['sj_rank']) - as_float(row['af3_rank_from_score'])

target_sizes = np.array([len(v) for v in targets.values()])
print(f'rows: {len(rows):,}')
print(f'targets: {len(targets):,}')
print(f'samples per target: min={target_sizes.min()}, median={np.median(target_sizes):.0f}, max={target_sizes.max()}')
print(f'native paths: {len(set(r["native_path"] for r in rows)):,}')
print(f'model paths: {len(set(r["path"] for r in rows)):,}')

for col in ['pred_score', 'af3_ranking_score', 'sj_rank', 'af3_rank_from_score', 'rank_delta_sj_minus_af3_score']:
    arr = finite_array(rows, col)
    print(f'{col:24s} n={len(arr):6d} mean={arr.mean():8.3f} median={np.median(arr):8.3f} p05={np.percentile(arr, 5):8.3f} p95={np.percentile(arr, 95):8.3f}')

for col in METRIC_COLS:
    n_missing = sum(not np.isfinite(as_float(r.get(col))) for r in rows)
    print(f'missing {col}: {n_missing:,} / {len(rows):,}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)

axes[0, 0].hist(finite_array(rows, 'pred_score'), bins=80, color='#3b6ea8')
axes[0, 0].set_title('SJscore distribution')
axes[0, 0].set_xlabel('pred_score; lower ranks first')
axes[0, 0].set_ylabel('models')

axes[0, 1].hist(finite_array(rows, 'af3_ranking_score'), bins=80, color='#4b9b6f')
axes[0, 1].set_title('AF3 ranking score distribution')
axes[0, 1].set_xlabel('af3_ranking_score; higher ranks first')

axes[0, 2].hist(target_sizes, bins=np.arange(target_sizes.min(), target_sizes.max() + 2) - 0.5, color='#7d6ab2')
axes[0, 2].set_title('Samples per target')
axes[0, 2].set_xlabel('rows')

sj_rank = finite_array(rows, 'sj_rank')
af3_score_rank = finite_array(rows, 'af3_rank_from_score')
hb = axes[1, 0].hexbin(sj_rank, af3_score_rank, gridsize=45, mincnt=1, cmap='viridis')
axes[1, 0].plot([1, 100], [1, 100], color='black', lw=1, alpha=0.6)
axes[1, 0].set_title('SJscore rank vs AF3 score-derived rank')
axes[1, 0].set_xlabel('SJscore rank; lower is better')
axes[1, 0].set_ylabel('AF3 score-derived rank; lower is better')
fig.colorbar(hb, ax=axes[1, 0], label='models')

axes[1, 1].hist(finite_array(rows, 'rank_delta_sj_minus_af3_score'), bins=80, color='#b07a3f')
axes[1, 1].axvline(0, color='black', lw=1)
axes[1, 1].set_title('Rank delta')
axes[1, 1].set_xlabel('SJscore rank - AF3 score-derived rank')

rank_corr = []
for target_rows in targets.values():
    x = np.array([as_float(r['sj_rank']) for r in target_rows], dtype=float)
    y = np.array([as_float(r['af3_rank_from_score']) for r in target_rows], dtype=float)
    corr = spearmanr(x, y, nan_policy='omit').correlation
    if np.isfinite(corr):
        rank_corr.append(corr)
rank_corr = np.array(rank_corr)
axes[1, 2].hist(rank_corr, bins=40, color='#8a4f7d')
axes[1, 2].axvline(np.median(rank_corr), color='black', lw=1, label=f'median={np.median(rank_corr):.2f}')
axes[1, 2].set_title('Per-target rank agreement')
axes[1, 2].set_xlabel('Spearman(SJscore rank, AF3 score-derived rank)')
axes[1, 2].legend(frameon=False)

fig.savefig(PLOT_DIR / 'score_and_rank_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
def top_n_per_target(all_rows, score_col, n, reverse=False):
    selected = []
    for target_rows in group_rows(all_rows).values():
        selected.extend(sorted(target_rows, key=lambda r: as_float(r[score_col]), reverse=reverse)[:n])
    return selected


def select_rows_for_metric_calculation(all_rows):
    if COMPUTE_ALL_METRICS:
        selected = list(all_rows)
    else:
        selected = []
        selected.extend(top_n_per_target(all_rows, 'pred_score', TOP_N_FOR_METRICS, reverse=False))
        selected.extend(top_n_per_target(all_rows, 'af3_ranking_score', TOP_N_FOR_METRICS, reverse=True))

    dedup = {}
    for row in selected:
        row['metric_key'] = metric_key(row)
        dedup[row['metric_key']] = row
    return list(dedup.values())


def read_metric_cache(path):
    if not path.exists():
        return {}
    with path.open(newline='') as f:
        cache_rows = list(csv.DictReader(f))
    cache = {}
    for row in cache_rows:
        if 'metric_key' not in row or not row['metric_key']:
            row['metric_key'] = metric_key(row)
        for col in METRIC_COLS:
            row[col] = as_float(row.get(col))
        cache[row['metric_key']] = row
    return cache


def write_metric_cache(path, cache):
    fieldnames = ['metric_key', 'target_id', 'sample_id', 'native_path', 'path', *METRIC_COLS, 'metric_error']
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in cache.values():
            writer.writerow({k: row.get(k, '') for k in fieldnames})


def _compute_metrics_for_target(payload):
    target_id, target_rows = payload

    from evaluation.loop_metrics import compute_loop_metrics_from_structures, load_structure
    from evaluation.docking_metrics import compute_dockq_style_metrics_from_structures

    native_cache = {}
    out_rows = []
    for rec in target_rows:
        out = {
            'metric_key': rec['metric_key'],
            'target_id': rec['target_id'],
            'sample_id': rec['sample_id'],
            'native_path': rec['native_path'],
            'path': rec['path'],
            'loop_rmsd': np.nan,
            'loop_lddt': np.nan,
            'irmsd': np.nan,
            'lrmsd': np.nan,
            'metric_error': '',
        }
        try:
            native_path = rec['native_path']
            if native_path not in native_cache:
                native_cache[native_path] = load_structure(native_path)
            native_structure = native_cache[native_path]
            model_structure = load_structure(rec['path'])
            loop = compute_loop_metrics_from_structures(
                native_structure,
                model_structure,
                native_path=native_path,
                model_path=rec['path'],
                rmsd_atom_type='backbone',
            )
            dock = compute_dockq_style_metrics_from_structures(
                native_structure,
                model_structure,
                native_path=native_path,
                model_path=rec['path'],
            )
            out['loop_rmsd'] = loop.loop_rmsd
            out['loop_lddt'] = loop.loop_lddt
            out['irmsd'] = dock.irmsd
            out['lrmsd'] = dock.lrmsd
        except Exception as exc:
            out['metric_error'] = f'{type(exc).__name__}: {exc}'[:500]
        out_rows.append(out)
    return out_rows


selected = select_rows_for_metric_calculation(rows)
cache = read_metric_cache(CACHE_PATH)
missing = [row for row in selected if row['metric_key'] not in cache]
print(f'selected rows for metric calculation: {len(selected):,} / {len(rows):,}')
print(f'cached rows: {len(cache):,}; missing selected rows: {len(missing):,}')

if missing:
    jobs = [(target_id, target_rows) for target_id, target_rows in group_rows(missing).items()]
    start = time.time()
    completed = 0
    if MAX_WORKERS <= 1:
        for job in jobs:
            for row in _compute_metrics_for_target(job):
                cache[row['metric_key']] = row
            completed += 1
            if completed % 10 == 0 or completed == len(jobs):
                print(f'completed {completed}/{len(jobs)} targets')
    else:
        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(_compute_metrics_for_target, job) for job in jobs]
            for fut in as_completed(futures):
                for row in fut.result():
                    cache[row['metric_key']] = row
                completed += 1
                if completed % 10 == 0 or completed == len(jobs):
                    print(f'completed {completed}/{len(jobs)} targets')
    write_metric_cache(CACHE_PATH, cache)
    print(f'wrote cache: {CACHE_PATH} ({len(cache):,} rows, {time.time() - start:.1f}s)')
else:
    print('no new metric rows needed')

errors = [r for r in cache.values() if r.get('metric_error')]
print(f'metric errors in cache: {len(errors):,}')
for row in errors[:10]:
    print(row['target_id'], row['sample_id'], row['metric_error'])

In [ ]:
cache = read_metric_cache(CACHE_PATH)
analysis_rows = []
for row in rows:
    out = dict(row)
    out['metric_key'] = metric_key(row)
    cached = cache.get(out['metric_key'])
    if cached:
        for col in METRIC_COLS:
            out[col] = as_float(cached.get(col))
    analysis_rows.append(out)

metric_rows = [r for r in analysis_rows if np.isfinite(as_float(r.get('loop_rmsd'))) and np.isfinite(as_float(r.get('loop_lddt')))]
print(f'rows with loop metrics available: {len(metric_rows):,} / {len(analysis_rows):,}')
if len(metric_rows) < len(analysis_rows):
    print('WARNING: metric-backed plots below are using a partial metric cache. Run the metric calculation cell with COMPUTE_ALL_METRICS=True to cover all decoys.')
for col in METRIC_COLS:
    print(f'{col:10s} computed rows: {len(finite_array(analysis_rows, col)):,}')

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, metric, color in zip(axes.ravel(), METRIC_COLS, ['#3b6ea8', '#4b9b6f', '#b07a3f', '#8a4f7d']):
    values = finite_array(metric_rows, metric)
    ax.hist(values, bins=60, color=color)
    ax.set_title(metric)
    ax.set_xlabel(metric)
    ax.set_ylabel('models')
fig.savefig(PLOT_DIR / 'computed_metric_distributions.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for row_idx, (score_col, score_label) in enumerate([('pred_score', 'SJscore; lower is better'), ('af3_ranking_score', 'AF3 ranking score; higher is better')]):
    for col_idx, (metric, metric_label) in enumerate([('loop_rmsd', 'Loop RMSD'), ('loop_lddt', 'Loop lDDT')]):
        ax = axes[row_idx, col_idx]
        x = np.array([as_float(r[score_col]) for r in metric_rows], dtype=float)
        y = np.array([as_float(r[metric]) for r in metric_rows], dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        hb = ax.hexbin(x[mask], y[mask], gridsize=45, mincnt=1, cmap='magma')
        ax.set_xlabel(score_label)
        ax.set_ylabel(metric_label)
        ax.set_title(f'{score_col} vs {metric}')
        fig.colorbar(hb, ax=ax, label='models')
fig.savefig(PLOT_DIR / 'score_vs_loop_metrics.png', bbox_inches='tight')
plt.show()

In [ ]:
def best_metric_value(values, lower_is_better):
    arr = np.array([as_float(v) for v in values], dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan
    return float(np.min(arr) if lower_is_better else np.max(arr))


def topk_metric_table(all_metric_rows, k_values=(1, 5, 10)):
    specs = {
        'SJscore': ('pred_score', False),
        'AF3 ranking score': ('af3_ranking_score', True),
    }
    table = []
    grouped = group_rows(all_metric_rows)
    for method, (score_col, reverse) in specs.items():
        for target_id, target_rows in grouped.items():
            ordered = sorted(target_rows, key=lambda r: as_float(r[score_col]), reverse=reverse)
            for k in k_values:
                topk = ordered[:k]
                out = {'target_id': target_id, 'method': method, 'top_k': k}
                for metric in METRIC_COLS:
                    out[metric] = best_metric_value([r.get(metric) for r in topk], LOWER_IS_BETTER[metric])
                table.append(out)
    return table


topk_perf = topk_metric_table(metric_rows)
for method in ['SJscore', 'AF3 ranking score']:
    for k in [1, 5, 10]:
        sub = [r for r in topk_perf if r['method'] == method and r['top_k'] == k]
        parts = []
        for metric in METRIC_COLS:
            arr = finite_array(sub, metric)
            parts.append(f'{metric} median={np.median(arr):.3f} mean={np.mean(arr):.3f} n={len(arr)}')
        print(f'{method:18s} top{k:2d}: ' + ' | '.join(parts))

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
methods = ['SJscore', 'AF3 ranking score']
colors = ['#3b6ea8', '#4b9b6f']
for ax, metric in zip(axes.ravel(), METRIC_COLS):
    positions, labels, data = [], [], []
    pos = 1
    for k in [1, 5, 10]:
        for method in methods:
            values = finite_array([r for r in topk_perf if r['top_k'] == k and r['method'] == method], metric)
            data.append(values)
            positions.append(pos)
            labels.append(f'{method}\ntop{k}')
            pos += 1
        pos += 0.8
    bp = ax.boxplot(data, positions=positions, widths=0.65, patch_artist=True, showfliers=False)
    for patch, i in zip(bp['boxes'], range(len(bp['boxes']))):
        patch.set_facecolor(colors[i % len(methods)])
        patch.set_alpha(0.75)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_title(f'Top-k best {metric}')
    ax.set_ylabel(metric)
fig.savefig(PLOT_DIR / 'sjscore_vs_af3_topk_metric_performance.png', bbox_inches='tight')
plt.show()

In [ ]:
corr_rows = []
for target_id, target_rows in group_rows(metric_rows).items():
    if len(target_rows) < 3:
        continue
    for metric in METRIC_COLS:
        y = np.array([as_float(r[metric]) for r in target_rows], dtype=float)
        corr_rows.append({
            'target_id': target_id,
            'metric': metric,
            'sjscore_spearman': spearmanr([as_float(r['pred_score']) for r in target_rows], y, nan_policy='omit').correlation,
            'af3_score_spearman': spearmanr([as_float(r['af3_ranking_score']) for r in target_rows], y, nan_policy='omit').correlation,
        })

for metric in METRIC_COLS:
    sub = [r for r in corr_rows if r['metric'] == metric]
    print(metric)
    for col in ['sjscore_spearman', 'af3_score_spearman']:
        arr = finite_array(sub, col)
        print(f'  {col:20s} median={np.median(arr):7.3f} mean={np.mean(arr):7.3f} n={len(arr)}')

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
corr_cols = ['sjscore_spearman', 'af3_score_spearman']
for ax, metric in zip(axes.ravel(), METRIC_COLS):
    sub = [r for r in corr_rows if r['metric'] == metric]
    data = [finite_array(sub, col) for col in corr_cols]
    ax.boxplot(data, labels=['SJscore', 'AF3 score'], showfliers=False)
    ax.axhline(0, color='black', lw=1)
    ax.set_ylim(-1.05, 1.05)
    ax.set_title(f'Per-target Spearman vs {metric}')
    ax.set_ylabel('Spearman correlation')
fig.savefig(PLOT_DIR / 'per_target_score_metric_correlations.png', bbox_inches='tight')
plt.show()

top1 = [r for r in topk_perf if r['top_k'] == 1]
top1_by_target = defaultdict(dict)
for row in top1:
    top1_by_target[row['target_id']][row['method']] = row

success_rows = []
for method in ['SJscore', 'AF3 ranking score']:
    method_rows = [r for r in top1 if r['method'] == method]
    usable = []
    for row in method_rows:
        loop_rmsd = as_float(row.get('loop_rmsd'))
        loop_lddt = as_float(row.get('loop_lddt'))
        if np.isfinite(loop_rmsd) or np.isfinite(loop_lddt):
            rmsd_success = np.isfinite(loop_rmsd) and loop_rmsd <= SUCCESS_RMSD_CUTOFF
            lddt_success = np.isfinite(loop_lddt) and loop_lddt >= SUCCESS_LDDT_CUTOFF
            usable.append((rmsd_success, lddt_success, rmsd_success or lddt_success))
    success_rows.append({
        'method': method,
        'n_targets': len(usable),
        'rmsd_success_rate': np.mean([x[0] for x in usable]) if usable else np.nan,
        'lddt_success_rate': np.mean([x[1] for x in usable]) if usable else np.nan,
        'loop_success_rate': np.mean([x[2] for x in usable]) if usable else np.nan,
    })

for row in success_rows:
    print(
        f"{row['method']:18s} n={row['n_targets']:3d} "
        f"success={(100 * row['loop_success_rate']):5.1f}% "
        f"RMSD<={SUCCESS_RMSD_CUTOFF:g}A={(100 * row['rmsd_success_rate']):5.1f}% "
        f"lDDT>={SUCCESS_LDDT_CUTOFF:g}={(100 * row['lddt_success_rate']):5.1f}%"
    )

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
methods = [r['method'] for r in success_rows]
combined = [100 * r['loop_success_rate'] for r in success_rows]
axes[0].bar(methods, combined, color=['#3b6ea8', '#4b9b6f'])
axes[0].set_ylim(0, 100)
axes[0].set_ylabel('success rate (%)')
axes[0].set_title(f'Top1 loop success\nRMSD <= {SUCCESS_RMSD_CUTOFF:g}A or lDDT >= {SUCCESS_LDDT_CUTOFF:g}')
for i, value in enumerate(combined):
    axes[0].text(i, value + 1, f'{value:.1f}%', ha='center', va='bottom')

x = np.arange(len(methods))
width = 0.36
rmsd_rates = [100 * r['rmsd_success_rate'] for r in success_rows]
lddt_rates = [100 * r['lddt_success_rate'] for r in success_rows]
axes[1].bar(x - width / 2, rmsd_rates, width=width, color='#7d6ab2', label=f'RMSD <= {SUCCESS_RMSD_CUTOFF:g}A')
axes[1].bar(x + width / 2, lddt_rates, width=width, color='#b07a3f', label=f'lDDT >= {SUCCESS_LDDT_CUTOFF:g}')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].set_ylim(0, 100)
axes[1].set_ylabel('success rate (%)')
axes[1].set_title('Component success rates')
axes[1].legend(frameon=False)
fig.savefig(PLOT_DIR / 'top1_loop_success_rate_sjscore_vs_af3_score.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)
for ax, metric in zip(axes.ravel(), METRIC_COLS):
    xs, ys = [], []
    for method_rows in top1_by_target.values():
        if 'SJscore' in method_rows and 'AF3 ranking score' in method_rows:
            x = as_float(method_rows['AF3 ranking score'][metric])
            y = as_float(method_rows['SJscore'][metric])
            if np.isfinite(x) and np.isfinite(y):
                xs.append(x)
                ys.append(y)
    xs = np.array(xs)
    ys = np.array(ys)
    ax.scatter(xs, ys, s=18, alpha=0.65, color='#3b6ea8')
    lo = min(xs.min(), ys.min())
    hi = max(xs.max(), ys.max())
    ax.plot([lo, hi], [lo, hi], color='black', lw=1)
    better = np.mean(ys < xs) if LOWER_IS_BETTER[metric] else np.mean(ys > xs)
    ax.set_title(f'Top1 SJscore vs AF3 score: {metric}\nSJscore better on {better:.1%} of targets')
    ax.set_xlabel(f'AF3 top1 {metric}')
    ax.set_ylabel(f'SJscore top1 {metric}')
fig.savefig(PLOT_DIR / 'paired_top1_sjscore_vs_af3.png', bbox_inches='tight')
plt.show()

print(f'plots saved under: {PLOT_DIR}')
print(f'metric cache: {CACHE_PATH}')